In [1]:
import sys
sys.path.append('../../research/berry/')

In [2]:
import fast_inla
import numpy as np

/Users/tbent/.mambaforge/envs/kevlar/lib/python3.10/site-packages/jax/_src/lib/__init__.py:33: UserWarning: JAX on Mac ARM machines is experimental and minimally tested. Please see https://github.com/google/jax/issues/5501 in the event of problems.
  warnings.warn("JAX on Mac ARM machines is experimental and minimally tested. "


In [17]:
from pykevlar.core.model import ModelBase, SimGlobalStateBase, SimStateBase
from pykevlar.core.model.binomial import SimpleSelection, SimpleSelectionSimGlobalState, SimpleSelectionSimState


class KevlarBerryINLA(SimpleSelection):
    def __init__(self, n_arms, n_arm_samples, gr):
        super().__init__(n_arms, n_arm_samples, 0, [2.1])
        # self.critical_values([2.1])
        self.gr = gr

class SGS(SimGlobalStateBase):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def make_sim_state(self):
        print("making sim state")
        return SS(self.model)

class SS(SimStateBase):
    def __init__(self, model):
        self.model = model

    def simulate(self, gen, rej_len):
        print(gen, rej_len)
        return None


class BS(SimpleSelectionSimState):
    pass

In [19]:
SimpleSelectionSimState()

TypeError: pykevlar.core.model.binomial.SimpleSelectionSimState: No constructor defined!

In [12]:
n_arms = 2
arm_samples = 35
n_sims = 10
seed = 10

In [13]:
from pykevlar.grid import HyperPlane

# define null hypos
null_hypos = []
for i in range(1, n_arms):
    n = np.zeros(n_arms)
    n[0] = 1
    n[i] = -1
    null_hypos.append(HyperPlane(n, 0))

from utils import make_cartesian_grid_range

gr = make_cartesian_grid_range(3, np.full(n_arms, -0.5), np.full(n_arms, 0.5), 1000)
gr.create_tiles(null_hypos)
gr.prune()

In [14]:
from pykevlar.core.driver import accumulate
from pykevlar.core.bound import TypeIErrorAccum
model = KevlarBerryINLA(n_arms, arm_samples, gr)
sgs = SGS(model)

In [15]:
# prepare output
acc_o = TypeIErrorAccum(
    model.n_models(),
    gr.n_tiles(),
    gr.n_params()
)

model.simulate()
acc_o.update(rej_len, )

In [16]:

# run C++ core routine
accumulate(
    sim_global_state=sgs,
    grid_range=gr,
    accum=acc_o,
    sim_size=n_sims,
    seed=10,
    n_threads=1
)

HI
HI
HI
